In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from zuko.flows import UnconditionalDistribution
from torch.distributions import Cauchy, Normal, Laplace, Bernoulli, Uniform
from causalflows.flows import CausalFlow
import copy
from causal_cocycle.ssvkernel import ssvkernel
import numpy as np
from scipy.stats import betaprime, norm
from architectures import get_nsf_transforms

In [2]:
# ── Data ─────────────────────────────────────────────────────
def draw_abs_nbp(size):
    """Draw |Normal-Beta-Prime(0.1, 0.1)| samples."""
    tau = betaprime.rvs(0.1, 0.1, size=size)
    values = norm.rvs(scale=np.sqrt(tau))
    return np.abs(values)


class MixedTails:
    def sample(self, size):
        selector = Bernoulli(0.5).sample(size)
        positive = Normal(0, 1).sample(size).abs()
        negative = torch.as_tensor(
            draw_abs_nbp(size), dtype=positive.dtype
        )
        return positive * selector - negative * (1 - selector)


seed = 0
truth_seed = 2026
torch.manual_seed(seed)
np.random.seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

N_train_per_treatment = 1_000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Y0 = MixedTails().sample((N_train_per_treatment, 1))
Y1 = 2 * Y0

d = 1
base0 = UnconditionalDistribution(
    Normal, loc=torch.zeros(d), scale=torch.ones(d), buffer=True
)
base1 = UnconditionalDistribution(
    Normal, loc=torch.zeros(d), scale=torch.ones(d), buffer=True
)
nsf0, nsf1 = get_nsf_transforms()
flow0 = CausalFlow(transform=nsf0, base=base0).to(device)
flow1 = CausalFlow(transform=nsf1, base=base1).to(device)


In [3]:
from torch.optim import Adam

learn_rate = 1e-2
batch_size = 128
epochs = 1_000

loader0 = DataLoader(
    TensorDataset(Y0), batch_size=batch_size, shuffle=True
)
loader1 = DataLoader(
    TensorDataset(Y1), batch_size=batch_size, shuffle=True
)
opt0 = Adam(flow0.parameters(), lr=learn_rate)
opt1 = Adam(flow1.parameters(), lr=learn_rate)

training_losses = []
for epoch in range(1, epochs + 1):
    running0 = 0.0
    for (y_batch,) in loader0:
        y_batch = y_batch.to(device)
        loss0 = -flow0().log_prob(y_batch).mean()
        opt0.zero_grad()
        loss0.backward()
        opt0.step()
        running0 += loss0.item() * y_batch.size(0)

    running1 = 0.0
    for (y_batch,) in loader1:
        y_batch = y_batch.to(device)
        loss1 = -flow1().log_prob(y_batch).mean()
        opt1.zero_grad()
        loss1.backward()
        opt1.step()
        running1 += loss1.item() * y_batch.size(0)

    mean_losses = (
        running0 / N_train_per_treatment,
        running1 / N_train_per_treatment,
    )
    training_losses.append(mean_losses)
    if epoch == 1 or epoch % 100 == 0:
        print(
            f"Epoch {epoch:4d} | flow0 NLL = {mean_losses[0]:.3f} "
            f"| flow1 NLL = {mean_losses[1]:.3f}"
        )


Epoch    1 | flow0 NLL = 3141064519233320972731582628495360.000 | flow1 NLL = 958350033569218222679370646945792.000


Epoch  100 | flow0 NLL = 3141064479619242445717536740737024.000 | flow1 NLL = 958350033569217502103430267666432.000


Epoch  200 | flow0 NLL = 3141064479619242445717536740737024.000 | flow1 NLL = 958350058328019031445406237065216.000


Epoch  300 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350033569218222679370646945792.000


Epoch  400 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350033569217502103430267666432.000


...

Epoch  600 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350058328018310869465857785856.000


Epoch  700 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350033569218222679370646945792.000


Epoch  800 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350033569217502103430267666432.000


Epoch  900 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350033569218222679370646945792.000


Epoch 1000 | flow0 NLL = 3141064519233323278574591842189312.000 | flow1 NLL = 958350058328019031445406237065216.000


In [4]:
# ── Abduct, act, and predict ─────────────────────────────────
N_effect = 100_000
effect_seed = 17
torch.manual_seed(effect_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(effect_seed)

flow0.eval()
flow1.eval()
with torch.no_grad():
    latent = flow0().base.sample((N_effect,))
    Y0_eff = flow0().transform.inv(latent).cpu().reshape(-1)
    Y1_eff = flow1().transform.inv(latent).cpu().reshape(-1)
delta = Y1_eff - Y0_eff

torch.manual_seed(truth_seed)
np.random.seed(truth_seed)
true_delta = MixedTails().sample((N_effect,)).cpu()


In [5]:
from scm_misspecification_figure import (
    RESULT_PATHS,
    plot_result_preview,
    save_effect_results,
)

result_path = save_effect_results(
    RESULT_PATHS[("mixed_tails", "gaussian_base")],
    estimated_effect=delta,
    true_effect=true_delta,
    method="gaussian_base",
    noise="mixed_tails",
    seed=seed,
    truth_seed=truth_seed,
    n_train_per_treatment=N_train_per_treatment,
    n_effect=N_effect,
    epochs=epochs,
    learning_rate=learn_rate,
)
print(f"Saved {result_path}")
fig, ax = plot_result_preview(result_path)
plt.show()


Saved /nfs/ghome/live/danceh/Cocycles/examples/scm_example/paper_results/mixed_tails_gaussian_base.npz
